# Partie 3 - Representation des donnees

On repart des donnees deja nettoyees dans le notebook precedent pour les transformer en vecteurs numeriques, seule forme que les modeles de machine learning peuvent utiliser.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from text_cleaning import clean_text

df = pd.read_csv("../data/raw/tweets_suspect.csv")
df = df.dropna(subset=["message", "label"]).drop_duplicates(subset=["message"])
df["clean_text"] = df["message"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0]

print(df.shape)


(59416, 3)


## Pourquoi le TF-IDF

Le sujet propose plusieurs facons de transformer le texte en nombres, des approches simples comme Bag of Words ou TF-IDF, et des approches plus avancees comme Word2Vec ou BERT. On a choisi le **TF-IDF** pour plusieurs raisons :

- Les tweets sont courts, donc pas besoin d'une representation tres complexe pour capturer le sens general.
- TF-IDF ne se contente pas de compter les mots comme le Bag of Words classique : il diminue le poids des mots qui reviennent dans presque tous les tweets, et augmente celui des mots plus specifiques, ce qui aide a mieux distinguer les tweets suspects des autres.
- C'est rapide a calculer et facile a comprendre, contrairement a des embeddings comme BERT qui demandent beaucoup plus de ressources et de temps d'entrainement.
- C'est une bonne base de depart : si les resultats ne sont pas satisfaisants, on pourra toujours essayer une methode plus avancee ensuite.

On a laisse la possibilite d'utiliser des bigrammes en plus des mots seuls (parametre `ngram_max` dans `params.yaml`), pour capturer un peu de contexte (par exemple "not good" est different de "good").

In [2]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["clean_text"])

print("nombre de tweets :", X.shape[0])
print("taille du vocabulaire :", X.shape[1])


nombre de tweets : 59416
taille du vocabulaire : 20000


## Un exemple concret

Pour voir ce que fait le TF-IDF, on regarde les mots avec le score le plus eleve pour quelques tweets.

In [3]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()

for i in range(3):
    row = X[i].toarray().flatten()
    top_idx = row.argsort()[-5:][::-1]
    mots_importants = [feature_names[j] for j in top_idx if row[j] > 0]
    print("tweet :", df["clean_text"].iloc[i])
    print("mots les plus importants selon TF-IDF :", mots_importants)
    print("-" * 60)


tweet : awww bummer shoulda got david carr third day
mots les plus importants selon TF-IDF : ['carr', 'shoulda', 'third', 'david', 'bummer']
------------------------------------------------------------
tweet : upset update facebook texting might cry result school today also blah
mots les plus importants selon TF-IDF : ['today also', 'update facebook', 'texting', 'school today', 'blah']
------------------------------------------------------------
tweet : dived many time ball managed save rest go bound
mots les plus importants selon TF-IDF : ['bound', 'many time', 'managed', 'ball', 'save']
------------------------------------------------------------


## Ce qu'on retient

Le TF-IDF transforme chaque tweet en un vecteur de plusieurs milliers de dimensions (une par mot ou paire de mots du vocabulaire), ou chaque valeur represente l'importance du mot dans ce tweet par rapport a l'ensemble des tweets. C'est cette representation qui sert d'entree aux modeles entraines dans `src/train.py` et qui sera reprise dans la partie suivante pour comparer plusieurs algorithmes de classification.